In [53]:
import pandas as pd

In [54]:
pd.__version__

'2.3.0'

In [36]:
# Load dataset
df = pd.read_csv("C:/Users/sanka/Downloads/project1/data/raw/healthcare_fraud_detection.csv")
df.head()

,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.0
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.0
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.0
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.0
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.0


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Provider_ID                            10000 non-null  object 
 1   Claim_ID                               10000 non-null  object 
 2   Patient_Age                            10000 non-null  int64  
 3   Patient_Gender                         10000 non-null  object 
 4   Diagnosis_Code                         10000 non-null  object 
 5   Procedure_Code                         10000 non-null  int64  
 6   Claim_Amount                           10000 non-null  float64
 7   Approved_Amount                        10000 non-null  float64
 8   Insurance_Type                         9650 non-null   object 
 9   Claim_Submission_Date                  10000 non-null  object 
 10  Days_Between_Service_and_Claim         10000 non-null  int64  
 11  Num

In [38]:
# Notice Insurance_Type, Provider_Specialty and Prior_Visits_12m are missing data so we fill them in
df["Insurance_Type"] = df["Insurance_Type"].fillna("Unknown")
df["Provider_Specialty"] = df["Provider_Specialty"].fillna("Unknown")

In [39]:
df["Prior_Visits_12m"] = df["Prior_Visits_12m"].fillna(df["Prior_Visits_12m"].median())

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Provider_ID                            10000 non-null  object 
 1   Claim_ID                               10000 non-null  object 
 2   Patient_Age                            10000 non-null  int64  
 3   Patient_Gender                         10000 non-null  object 
 4   Diagnosis_Code                         10000 non-null  object 
 5   Procedure_Code                         10000 non-null  int64  
 6   Claim_Amount                           10000 non-null  float64
 7   Approved_Amount                        10000 non-null  float64
 8   Insurance_Type                         10000 non-null  object 
 9   Claim_Submission_Date                  10000 non-null  object 
 10  Days_Between_Service_and_Claim         10000 non-null  int64  
 11  Num

In [41]:
# Assign the mean, standard deviation, minimum, maximum and the percentiles to a variable
global_claim_stats = df["Claim_Amount"].describe()
print(global_claim_stats)

count    10000.000000
mean       572.804406
std        406.202437
min         60.210000
25%        305.205000
50%        461.225000
75%        711.365000
max       6590.700000
Name: Claim_Amount, dtype: float64


In [42]:
# Group the dataset by the provider and calculate the mean claim amount for each one
provider_profiles = df.groupby("Provider_ID")["Claim_Amount"].mean().reset_index()

# Rename the columns
provider_profiles.rename(columns={"Claim_Amount": "Avg_Claim_Amount"}, inplace=True)

# Sort the providers from highest average billers to lowest
top_billers = provider_profiles.sort_values(by="Avg_Claim_Amount", ascending=False).head()
top_billers

,Provider_ID,Avg_Claim_Amount
101,P0101,823.385000
223,P0223,800.208864
289,P0289,798.465000
110,P0110,792.715778
78,P0078,750.073659


In [43]:
# The global mean and standard deviation
global_mean = df["Claim_Amount"].mean()
global_std = df["Claim_Amount"].std()

# The z-score for every single provider
provider_profiles["Z_Score"] = (provider_profiles["Avg_Claim_Amount"] - global_mean) / global_std

# Sort the table by the highest z-scores to see the most extreme outliers
anomalous_providers = provider_profiles.sort_values(by="Z_Score", ascending=False).head()
anomalous_providers

# TEST 1

,Provider_ID,Avg_Claim_Amount,Z_Score
101,P0101,823.385000,0.616886
223,P0223,800.208864,0.559830
289,P0289,798.465000,0.555537
110,P0110,792.715778,0.541384
78,P0078,750.073659,0.436406


In [44]:
#Since none of the z-scores are even above 1 standard deviation away from the mean, the data has no obvious anomalies.

In [45]:
# Find out what each provider charges for each specific procedure
provider_procedure_costs = df.groupby(["Provider_ID", "Procedure_Code"])["Claim_Amount"].mean().reset_index()
provider_procedure_costs.rename(columns={"Claim_Amount": "Provider_Avg_for_Procedure"}, inplace=True)

# Find the mean and standard deviation for each procedure across the whole dataset
global_procedure_costs = df.groupby("Procedure_Code")["Claim_Amount"].agg(
    Global_Proc_Mean='mean', 
    Global_Proc_Std='std'
).reset_index()

# Combine the tables together so we can compare them
merged_data = provider_procedure_costs.merge(global_procedure_costs, on="Procedure_Code")

# Calculate the z-score based on the specific procedure's normal cost, not the global cost
merged_data["Procedure_Z_Score"] = (merged_data["Provider_Avg_for_Procedure"] - merged_data["Global_Proc_Mean"]) / merged_data["Global_Proc_Std"]

# Sort to find the worst offenders
true_anomalies = merged_data.sort_values(by="Procedure_Z_Score", ascending=False).head()
true_anomalies

# TEST 2

,Provider_ID,Procedure_Code,Provider_Avg_for_Procedure,Global_Proc_Mean,Global_Proc_Std,Procedure_Z_Score
2535,P0295,99214,2640.48,595.727062,455.851451,4.485569
2307,P0269,80053,2328.34,572.402743,393.375814,4.463765
604,P0071,36415,2607.73,575.998039,478.595835,4.245194
9,P0001,36415,2221.22,575.998039,478.595835,3.437602
357,P0041,93000,1863.20,560.719433,402.594306,3.235219


In [46]:
# When we compared a provider's overall average to the global average, there were no anomalies.
# However, when we grouped by specific procedures, we found providers who systematically mark up their prices for those exact services.

In [47]:
# Our tests still fail if a provider has multiple normal claims for a procedure and very few abnormally high claims.
# This is because the normal claims smooth out the outliers, making the provider's average for a procedure seem legitimate.

In [48]:
# Merge the global procedure mean and standard deviation to specific claims
raw_claims = df.merge(global_procedure_costs, on="Procedure_Code")

# Find the z-score for each individual claims, instead of the average
raw_claims["Individual_Z_Score"] = (raw_claims["Claim_Amount"] - raw_claims["Global_Proc_Mean"]) / raw_claims["Global_Proc_Std"]

# Sort similarly
individual_anomalies = raw_claims.sort_values(by="Individual_Z_Score", ascending=False).head()
individual_anomalies[["Claim_ID", "Provider_ID", "Procedure_Code", "Claim_Amount", "Global_Proc_Mean", "Individual_Z_Score"]]

# TEST 3

,Claim_ID,Provider_ID,Procedure_Code,Claim_Amount,Global_Proc_Mean,Individual_Z_Score
507,C0000507,P0101,36415,6590.70,575.998039,12.567393
589,C0000589,P0039,93000,3657.89,560.719433,7.693031
8616,C0008616,P0151,97110,3624.82,570.694687,7.270151
2682,C0002682,P0286,99214,3901.94,595.727062,7.252830
5384,C0005384,P0289,93000,3469.03,560.719433,7.223924


In [55]:
# Even more claims that might've gone unnoticed in the previous few tests are caught when we compare each claim individually to the averages
# We can check the acuracy of each tests since we are told in the data if the claim was fraud or not in reality

eval_df = df[['Claim_ID', 'Provider_ID', 'Procedure_Code', 'Is_Fraud']].copy()

# Map test 1
eval_df = eval_df.merge(provider_profiles[['Provider_ID', 'Z_Score']], on='Provider_ID', how='left')

# Map test 2
eval_df = eval_df.merge(merged_data[['Provider_ID', 'Procedure_Code', 'Procedure_Z_Score']], on=['Provider_ID', 'Procedure_Code'], how='left')

# Map test 3
eval_df = eval_df.merge(raw_claims[['Claim_ID', 'Individual_Z_Score']], on='Claim_ID', how='left')